# F Việc 1 — Recipe Grid (≤6 arms) trên xLAM slice (Kaggle / Colab T4)

### Cell 1 — Thiết lập (ô chung, copy nguyên văn từ `notebooks/_setup_snippet.md`)


In [ ]:
# --- Thiết lập (Colab + local) --------------------------------------------------------------
# Colab : read GITHUB_TOKEN from Secrets, clone the private repo (skip if present), chdir into it.
# Local : walk up from the current directory until the repo root (docs/contracts/cli.md) is found.
# Sets REPO (Path), GIT_SHA, IN_COLAB, AUTHOR and a run() helper that calls repo scripts.
import os, shlex, subprocess, sys
from pathlib import Path

REPO_HTTPS = "github.com/thanhhao98/ChatSystem"
MARKER = "docs/contracts/cli.md"          # exists at the root of every checkout

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))


def _github_token():
    # Colab Secrets -> Kaggle Secrets -> environment variable. Never print the value.
    if IN_COLAB:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    return os.environ["GITHUB_TOKEN"]


if (IN_COLAB or IN_KAGGLE) and not Path(MARKER).exists():
    try:
        _token = _github_token()
    except Exception as e:  # SecretNotFoundError / NotebookAccessError / KeyError
        raise RuntimeError(
            "Thiếu secret GITHUB_TOKEN. Colab: biểu tượng chìa khoá (Secrets) -> Add new secret: "
            "Name = GITHUB_TOKEN, Value = Personal access token (classic, scope repo) của tài khoản collaborator "
            "trên thanhhao98/ChatSystem, bật 'Notebook access'. Kaggle: Add-ons -> Secrets -> GITHUB_TOKEN. "
            "Rồi chạy lại ô này.") from e
    if not Path("ChatSystem").exists():
        _r = subprocess.run(["git", "clone", "--quiet", f"https://{_token}@{REPO_HTTPS}", "ChatSystem"],
                            capture_output=True, text=True)
        if _r.returncode != 0:
            raise RuntimeError("git clone thất bại: " + _r.stderr.replace(_token, "<token>"))
    os.chdir("ChatSystem")
    del _token
else:
    _here = Path.cwd().resolve()
    for _cand in [_here, *_here.parents]:
        if (_cand / MARKER).exists():
            os.chdir(_cand)
            break
    else:
        raise FileNotFoundError(f"Không tìm thấy gốc repo (không có {MARKER}) khi đi lên từ {_here}. "
                                "Mở notebook từ bên trong thư mục ChatSystem đã clone.")

REPO = Path.cwd()


def _git(*args):
    # Small helper: run a git command in REPO and return stdout ("" on any failure).
    try:
        return subprocess.run(["git", *args], cwd=REPO, capture_output=True, text=True).stdout.strip()
    except OSError:
        return ""


GIT_SHA = _git("rev-parse", "--short", "HEAD") or "no-git"
AUTHOR = os.environ.get("GITHUB_USER") or _git("config", "user.name") or "điền tên"


def run(cmd):
    # Run a repo script (list of args), echo the command, stream its output, return CompletedProcess.
    shown = " ".join(shlex.quote(c) for c in cmd).replace(shlex.quote(sys.executable), "python", 1)
    print("$ " + shown)
    p = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
    if p.stdout:
        print(p.stdout.rstrip())
    if p.stderr:
        print(p.stderr.rstrip())
    print(f"[exit code = {p.returncode}]")
    return p


print(f"REPO      = {REPO}")
print(f"git HEAD  = {GIT_SHA}")
print(f"python    = {sys.version.split()[0]} · Colab = {IN_COLAB} · Kaggle = {IN_KAGGLE} · author = {AUTHOR}")

### Cell 2 — Cài thư viện và kiểm tra GPU
Phải thấy: `Tesla T4 · 15 GB · (7, 5) · USE_BF16=False → fp16`.
Nếu thấy `NO CUDA GPU`: *Runtime → Change runtime type → T4 GPU*.
Kaggle: dùng 1 GPU trong cấu hình T4×2 (script chỉ dùng 1 GPU).

In [ ]:
# Cell 2 — install the pinned training stack (Colab/Kaggle only) and probe the GPU.
import os
IN_KAGGLE = bool(globals().get("IN_KAGGLE")) or ((not IN_COLAB) and os.path.isdir("/kaggle/working"))
if IN_COLAB or IN_KAGGLE:
    !pip uninstall -y -q torchao
    !pip install -q -r requirements-train.txt
else:
    print("Local run: skipping pip install (use your own virtualenv built from requirements-train.txt).")

import importlib
import torch

FORCE_FP16 = os.environ.get("FORCE_FP16") == "1"
if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
    GPU_CAP = torch.cuda.get_device_capability(0)
    USE_BF16 = GPU_CAP[0] >= 8 and not FORCE_FP16
    GPU_LINE = (f"{GPU_NAME} · {GPU_GIB:.0f} GB · {GPU_CAP} · USE_BF16={USE_BF16} "
                f"→ {'bf16' if USE_BF16 else 'fp16'}")
else:
    GPU_NAME, GPU_GIB, GPU_CAP, USE_BF16 = None, 0.0, None, False
    GPU_LINE = "NO CUDA GPU (Runtime → Change runtime type → T4 GPU)"
print(GPU_LINE)

VERSIONS = {}
for _mod in ("torch", "transformers", "trl", "peft", "bitsandbytes", "accelerate"):
    try:
        VERSIONS[_mod] = importlib.import_module(_mod).__version__
    except Exception as exc:
        VERSIONS[_mod] = f"not installed ({type(exc).__name__})"
    print(f"{_mod:<13} {VERSIONS[_mod]}")
VERSIONS_LINE = " · ".join(f"{k} {v}" for k, v in VERSIONS.items())

### Cell 3 — Nơi lưu kết quả
Kaggle: `/kaggle/working/runs/t1_recipe_grid` — **mất khi phiên kết thúc**, tải về hoặc *Save Version* trước khi tắt.
Colab: Mount Google Drive. Drive path sống qua phiên; checkpoint để resume chỉ trong cùng phiên trên Kaggle.

In [ ]:
# Cell 3 — output directory. Keep RUN_NAME the same across all steps.
RUN_NAME = "t1_recipe_grid"   # một thư mục cho toàn bộ recipe grid experiment

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"/content/drive/MyDrive/ChatSystem/runs/{RUN_NAME}"
elif IN_KAGGLE:
    OUT = f"/kaggle/working/runs/{RUN_NAME}"
else:
    OUT = str(REPO / "runs" / RUN_NAME)
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT

# Tạo thư mục kết quả trong repo (nếu chưa có)
import subprocess
subprocess.run(["mkdir", "-p", "results/public"], cwd=str(REPO))

print("OUT =", OUT)
print("Arms sẽ chạy:", ["anchor", "r8", "lr2e4", "full_loss", "1p5b", "mask_fn"])

### Helper — VRAM monitor
Lấy mẫu `nvidia-smi` mỗi 5s để có VRAM đỉnh nếu phiên bị ngắt.

In [ ]:
import shlex, shutil, subprocess
_VRAM_PROC = None

def start_vram_monitor(log_path, every_s=5):
    global _VRAM_PROC
    stop_vram_monitor()
    if not shutil.which("nvidia-smi"):
        print("nvidia-smi not found; VRAM monitor disabled")
        return
    cmd = (f"while true; do nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits "
           f">> {shlex.quote(log_path)}; sleep {every_s}; done")
    _VRAM_PROC = subprocess.Popen(["bash", "-c", cmd])
    print(f"VRAM monitor -> {log_path}")

def stop_vram_monitor():
    global _VRAM_PROC
    if _VRAM_PROC is not None:
        _VRAM_PROC.terminate()
        try: _VRAM_PROC.wait(timeout=10)
        except subprocess.TimeoutExpired: _VRAM_PROC.kill()
        _VRAM_PROC = None

def peak_vram_gib(log_path):
    try:
        with open(log_path) as f:
            vals = [float(x) for x in f.read().split() if x.strip()]
    except FileNotFoundError:
        return None
    return max(vals) / 1024 if vals else None

print("helpers ready: start_vram_monitor / stop_vram_monitor / peak_vram_gib")

## Dry-run tất cả arms (CPU, không cần GPU)
Kiểm tra mọi arm trước khi tốn thời gian train. Phải thấy `over_limit=0` cho tất cả arms.
Chạy với `--recipe` để tái lập chính xác config sẽ dùng khi train thật.

In [ ]:
# Dry-run tất cả 6 arms — không tải model, chỉ tokenizer (~vài giây/arm)
ARMS = [
    "public_anchor",
    "public_r8",
    "public_lr2e4",
    "public_full_loss",
    "public_1p5b",
    "public_mask_fn",
]

for arm in ARMS:
    print(f"\n{'='*60}")
    print(f"DRY-RUN: {arm}")
    print('='*60)
    !python training/finetune_qlora.py \
        --recipe training/recipes/{arm}.yaml \
        --data data/public/xlam_2k.train.jsonl \
        --dry-run 2>&1

print("\nDry-run tất cả arms xong. Kiểm tra: mọi arm phải có over_limit=0")

## Train ARM 1: `public_anchor` ← CHẠY TRƯỚC, KIỂM TRA CI WEEK 2
**Bắt buộc:** Sau bước này, CI của anchor phải nằm trong (hoặc gần) Week 2 CI trước khi chạy arm khác.
Nếu thấp hơn nhiều → dừng, kiểm tra lại `--recipe` và `--completion-only`.

Config: 0.5B, r16/α32, lr 1e-4, completion_only=1, 1000 hàng, 1 epoch, seed 42.

In [ ]:
import os
os.makedirs(f"{OUT}/anchor", exist_ok=True)
start_vram_monitor(f"{OUT}/anchor/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_anchor.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/anchor \
    2>&1 | tee {OUT}/anchor/train.log
stop_vram_monitor()
print("anchor peak VRAM (nvidia-smi):", peak_vram_gib(f"{OUT}/anchor/vram.log"), "GiB")

In [ ]:
# Predict anchor trên eval set
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_anchor_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/anchor \
    2>&1 | tee {OUT}/anchor/predict.log

In [ ]:
# Eval anchor
!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_anchor_s42.pred.jsonl \
    --out  results/public/ft_public_anchor_s42.json \
    --md   results/public/ft_public_anchor_s42.md

# Bootstrap CI anchor (kiểm tra vs Week 2 CI)
print("\n--- Bootstrap CI anchor ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json

## Train ARM 2: `public_r8` (lora_rank=8, lora_alpha=16)
Trục thay đổi duy nhất: LoRA rank 16→8 (ít tham số hơn, có thể nhanh hơn, nhưng capacity thấp hơn).

In [ ]:
os.makedirs(f"{OUT}/r8", exist_ok=True)
start_vram_monitor(f"{OUT}/r8/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_r8.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/r8 \
    2>&1 | tee {OUT}/r8/train.log
stop_vram_monitor()
print("r8 peak VRAM:", peak_vram_gib(f"{OUT}/r8/vram.log"), "GiB")

In [ ]:
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_r8_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/r8

!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_r8_s42.pred.jsonl \
    --out  results/public/ft_public_r8_s42.json \
    --md   results/public/ft_public_r8_s42.md

print("\n--- Bootstrap CI: anchor vs r8 ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run r8=results/public/ft_public_r8_s42.json \
    --pair anchor r8

## Train ARM 3: `public_lr2e4` (lr=2e-4)
Trục thay đổi duy nhất: learning rate tăng gấp đôi (1e-4 → 2e-4).

In [ ]:
os.makedirs(f"{OUT}/lr2e4", exist_ok=True)
start_vram_monitor(f"{OUT}/lr2e4/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_lr2e4.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/lr2e4 \
    2>&1 | tee {OUT}/lr2e4/train.log
stop_vram_monitor()
print("lr2e4 peak VRAM:", peak_vram_gib(f"{OUT}/lr2e4/vram.log"), "GiB")

In [ ]:
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_lr2e4_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/lr2e4

!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_lr2e4_s42.pred.jsonl \
    --out  results/public/ft_public_lr2e4_s42.json \
    --md   results/public/ft_public_lr2e4_s42.md

print("\n--- Bootstrap CI: anchor vs lr2e4 ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run lr2e4=results/public/ft_public_lr2e4_s42.json \
    --pair anchor lr2e4

## Train ARM 4: `public_full_loss` (completion_only=0) — CHỈ QUAN SÁT
⚠️ **Arm này KHÔNG BAO GIỜ được chọn làm SGOD default** (theo spec F-V1).
Loss trên cả prompt làm model "học" format hệ thống thay vì nội dung tool call → thường thấp hơn anchor.

In [ ]:
os.makedirs(f"{OUT}/full_loss", exist_ok=True)
start_vram_monitor(f"{OUT}/full_loss/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_full_loss.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/full_loss \
    2>&1 | tee {OUT}/full_loss/train.log
stop_vram_monitor()
print("full_loss peak VRAM:", peak_vram_gib(f"{OUT}/full_loss/vram.log"), "GiB")

In [ ]:
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_full_loss_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/full_loss

!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_full_loss_s42.pred.jsonl \
    --out  results/public/ft_public_full_loss_s42.json \
    --md   results/public/ft_public_full_loss_s42.md

print("\n--- Bootstrap CI: anchor vs full_loss ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run full_loss=results/public/ft_public_full_loss_s42.json \
    --pair anchor full_loss

## Train ARM 5: `public_1p5b` (Qwen2.5-1.5B-Instruct)
Trục thay đổi duy nhất: model lớn hơn 3× (1.5B vs 0.5B). T4 15GB vẫn đủ với batch_size=4.
Thời gian train dài hơn anchor ~2-3× — ưu tiên chạy trên Kaggle (30 GPU-h/tuần).

In [ ]:
os.makedirs(f"{OUT}/1p5b", exist_ok=True)
start_vram_monitor(f"{OUT}/1p5b/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_1p5b.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/1p5b \
    2>&1 | tee {OUT}/1p5b/train.log
stop_vram_monitor()
print("1p5b peak VRAM:", peak_vram_gib(f"{OUT}/1p5b/vram.log"), "GiB")

In [ ]:
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_1p5b_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/1p5b

!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_1p5b_s42.pred.jsonl \
    --out  results/public/ft_public_1p5b_s42.json \
    --md   results/public/ft_public_1p5b_s42.md

print("\n--- Bootstrap CI: anchor vs 1p5b ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run 1p5b=results/public/ft_public_1p5b_s42.json \
    --pair anchor 1p5b

## Train ARM 6: `public_mask_fn` (mask_fn_names=0.5) — free arm
**Lý do chọn:** Mask 50% tên tool trong completion khi train buộc model học ngữ nghĩa tham số,
không chỉ nhớ tên tool → giả thuyết tăng `param_acc`. Chỉ đổi `mask_fn_names=0.5`, giữ nguyên mọi trục khác.

In [ ]:
os.makedirs(f"{OUT}/mask_fn", exist_ok=True)
start_vram_monitor(f"{OUT}/mask_fn/vram.log")
!python -u training/finetune_qlora.py \
    --recipe training/recipes/public_mask_fn.yaml \
    --data data/public/xlam_2k.train.jsonl \
    --val data/public/xlam_2k.val.jsonl \
    --output-dir {OUT}/mask_fn \
    2>&1 | tee {OUT}/mask_fn/train.log
stop_vram_monitor()
print("mask_fn peak VRAM:", peak_vram_gib(f"{OUT}/mask_fn/vram.log"), "GiB")

In [ ]:
!python training/predict_toolcall.py \
    --eval data/public/xlam_2k.eval.json \
    --out results/public/ft_public_mask_fn_s42.pred.jsonl \
    --backend transformers \
    --model-path {OUT}/mask_fn

!python training/eval_toolcall.py \
    --gold data/public/xlam_2k.eval.json \
    --pred results/public/ft_public_mask_fn_s42.pred.jsonl \
    --out  results/public/ft_public_mask_fn_s42.json \
    --md   results/public/ft_public_mask_fn_s42.md

print("\n--- Bootstrap CI: anchor vs mask_fn ---")
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run mask_fn=results/public/ft_public_mask_fn_s42.json \
    --pair anchor mask_fn

## Tổng hợp CI: tất cả arms vs anchor
Chạy lại tổng hợp để có bảng so sánh đầy đủ và McNemar cho mọi cặp.

In [ ]:
# Tổng hợp CI tất cả arms trong một lệnh
!python training/bootstrap_ci.py \
    --run anchor=results/public/ft_public_anchor_s42.json \
    --run r8=results/public/ft_public_r8_s42.json \
    --run lr2e4=results/public/ft_public_lr2e4_s42.json \
    --run full_loss=results/public/ft_public_full_loss_s42.json \
    --run 1p5b=results/public/ft_public_1p5b_s42.json \
    --run mask_fn=results/public/ft_public_mask_fn_s42.json \
    --pair anchor r8 \
    --pair anchor lr2e4 \
    --pair anchor full_loss \
    --pair anchor 1p5b \
    --pair anchor mask_fn \
    --md {OUT}/bootstrap_summary.md 2>&1 | tee {OUT}/bootstrap_summary.log

print("\nBootstrap summary saved to:", f"{OUT}/bootstrap_summary.md")

## Tạo báo cáo
Cell dưới đọc kết quả từ các file script ghi ra, in đúng khối Markdown để **dán vào bình luận task ClickUp F-V1**.
Không tự điền số — mọi mục chưa chạy hiện `n/a`.

In [ ]:
import datetime, json, re

def _read_json(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def _acc(path, key="accuracy"):
    d = _read_json(path)
    if d is None: return "n/a"
    v = d.get(key)
    return f"{v:.1f}" if isinstance(v, (int,float)) else "n/a"

def _cfg(arm_dir):
    d = _read_json(f"{arm_dir}/training_config.json")
    if d is None: return {}
    return d.get("resolved", {})

TODAY = datetime.date.today().isoformat()
ENV_LINE = "Colab" if IN_COLAB else ("Kaggle" if globals().get("IN_KAGGLE") else "local")
TASK = "F-V1 Recipe Grid ≤6 arms on xLAM slice"

ARM_DIRS = {
    "anchor":    f"{OUT}/anchor",
    "r8":        f"{OUT}/r8",
    "lr2e4":     f"{OUT}/lr2e4",
    "full_loss": f"{OUT}/full_loss",
    "1p5b":      f"{OUT}/1p5b",
    "mask_fn":   f"{OUT}/mask_fn",
}
RESULT_FILES = {
    "anchor":    "results/public/ft_public_anchor_s42.json",
    "r8":        "results/public/ft_public_r8_s42.json",
    "lr2e4":     "results/public/ft_public_lr2e4_s42.json",
    "full_loss": "results/public/ft_public_full_loss_s42.json",
    "1p5b":      "results/public/ft_public_1p5b_s42.json",
    "mask_fn":   "results/public/ft_public_mask_fn_s42.json",
}

lines = [
    f"## Báo cáo {TASK} — {TODAY} — {AUTHOR}",
    f"- Notebook: `notebooks/finetune/t1.ipynb` @ `{GIT_SHA}` · môi trường: {ENV_LINE}",
    f"- GPU: {GPU_LINE}",
    f"- Phiên bản: {VERSIONS_LINE}",
    f"- Data: `data/public/xlam_2k.train.jsonl` (1000 hàng đầu), eval: `xlam_2k.eval.json`, seed=42, epochs=1",
    "",
    "### Kết quả từng arm (strict_acc %)",
]

for arm, rfile in RESULT_FILES.items():
    d = _read_json(rfile)
    if d:
        acc = d.get('accuracy', 'n/a')
        tool_acc = d.get('tool_name_acc', 'n/a')
        param_acc = d.get('param_acc', 'n/a')
        cfg = _cfg(ARM_DIRS[arm])
        vram = cfg.get('peak_vram_gb', peak_vram_gib(f"{ARM_DIRS[arm]}/vram.log") or 'n/a')
        wall = cfg.get('wall_clock_min', 'n/a')
        lines.append(f"  - **{arm}**: strict_acc={acc:.1f if isinstance(acc,float) else acc}% "
                     f"| tool_name_acc={tool_acc:.1f if isinstance(tool_acc,float) else tool_acc}% "
                     f"| param_acc={param_acc:.1f if isinstance(param_acc,float) else param_acc}% "
                     f"| VRAM={vram} GB | wall={wall} min")
    else:
        lines.append(f"  - **{arm}**: n/a (chưa chạy)")

lines += [
    "",
    "### CI + McNemar (xem bootstrap_summary.log)",
    f"  - Bootstrap summary: `{OUT}/bootstrap_summary.log`",
    "  - **Quyết định:** CI của Δ (arm − anchor) bao gồm 0 → GIỮ anchor.",
    "  - **`full_loss` loại ngay** theo spec (completion_only=0 KHÔNG bao giờ là default).",
    "",
    "### Recipe được chọn: ĐIỀN SAU KHI ĐỌC CI",
    f"- PR: [F-V1] (điền link PR)",
    "",
    "### Lệnh đã chạy:",
    "```",
    "# Dry-run tất cả arms",
    "python training/finetune_qlora.py --recipe training/recipes/public_anchor.yaml \\",
    "    --data data/public/xlam_2k.train.jsonl --dry-run",
    "# Train anchor",
    f"python training/finetune_qlora.py --recipe training/recipes/public_anchor.yaml \\",
    f"    --data data/public/xlam_2k.train.jsonl --val data/public/xlam_2k.val.jsonl \\",
    f"    --output-dir {OUT}/anchor",
    "# (tương tự cho r8, lr2e4, full_loss, 1p5b, mask_fn)",
    "# Eval + CI",
    "python training/eval_toolcall.py --gold data/public/xlam_2k.eval.json \\",
    "    --pred results/public/ft_public_anchor_s42.pred.jsonl \\",
    "    --out results/public/ft_public_anchor_s42.json",
    "python training/bootstrap_ci.py --run anchor=results/public/ft_public_anchor_s42.json \\",
    "    --run r8=results/public/ft_public_r8_s42.json --pair anchor r8  # v.v.",
    "```",
]

print("\n".join(lines))